# **M608 Business Project in Computer Science: YouTube Channel Content Performance Dashboard**

**This Notebook processes the data retrieved directly from the YouTube Data API v3 to collect, aggregate and finally, analyze the performance metrics for the "visitBerlin" official channel.
The final outcome is an Interactive, Analytical Dashboard made with ipywidgets and Plotly.**

YouTube Channel Link: https://www.youtube.com/@visitBerlin

Project Demonstration Video Link: https://www.youtube.com/watch?v=jkpVVBP6B3E

# **1.INSTALLING THE LIBRARIES**

Here we are using the official Google API client (google-api-python-client) library for Python and it is used to communicate with the YouTube Data API v3. We also use pandas here for data manipulation and structured dataframe creation and this includes grouping, date parsing and the aggregation. Plotly is also used for creating the interactive visualizations and here we are using bar charts that respond to user selections. We also use the ipywidgets, which are the interactive UI elements which are used for dropdown menus and the layout boxes directly within the Jupyter-Colab Notebook.

In [1]:
#Installing the Libraries
!pip install google-api-python-client pandas plotly ipywidgets

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 31.6 MB/s eta 0:00:00


# **2. CONNECTING TO YOUTUBE API**

Now, this step authenticates our application with the Google Cloud services using the provided API key and the build function initializes the YouTube service object v3, which enables the endpoint queries for channel information, video list and any detailed video statistics.

In [2]:
# Authenticating and Building the YouTube API Service Object
from googleapiclient.discovery import build
API_KEY = "YOUR_API_KEY"
youtube = build("youtube", "v3", developerKey=API_KEY)
print("It's Connected Successfully, Cynthia!")

It's Connected Successfully, Cynthia!


# **3.FETCHING CHANNEL ID**

Each YouTube Channel has a unique Channel ID rather than a plain username for database querying and this request queries the channel endpoint using a custom username "visitBerlin" to dynamically retrieve its unique ID.

In [3]:
# Retrieving YouTube Channel ID for the username 'visitBerlin'
request = youtube.channels().list(part="id,snippet",forHandle="visitBerlin")
response = request.execute()
# Extracting the Main Channel's ID
channel_id = response["items"][0]["id"]
print("Main Channel ID:", channel_id)

Main Channel ID: UCeJu0ZtNt__ar0kSFum8V1Q


# **4.FETCHING RECENT VIDEOS**

We are using a specific Channel ID to request Data from the API, which is configured to filter for videos only and then sort them from newest to oldest and retrieve the details of the 50 Most Recently Uploaded Videos (Public Videos).

In [4]:
# Fetching the 50 Most Recently Uploaded Videos From YouTube Channel visitBerlin
request = youtube.search().list(part="snippet", channelId=channel_id, type="video", order="date", maxResults=50)
response = request.execute()
print("Cynthia! The Latest Videos have been Fetched Successfully!")

Cynthia! The Latest Videos have been Fetched Successfully!


# **5. EXTRACTING VIDEO METADATA TO DATAFRAME**

Here we are extracting the essential fields such as the title, the videoId, and the publishedAt from the API JSON response. So storing the parsed output in Pandas Dataframe allows Tabular Structured Analysis.

In [5]:
# Parsing Video Details into a Structured Pandas DataFrame
import pandas as pd
video_data = []
for item in response["items"]:
    video_data.append({
        "Video Title": item["snippet"]["title"],
        "Video ID": item["id"]["videoId"],
        "Published Date": item["snippet"]["publishedAt"][:10]
    })
videos_df = pd.DataFrame(video_data)
videos_df.head(50)

,Video Title,Video ID,Published Date
0,HIER IN BERLIN – Berlin und der Tourismus,krNFG3iActc,2018-03-05
1,HIER IN BERLIN – Berlin und der Tourismus,FaFP3JHGaWg,2018-03-05
2,Imagefilm Health Excellence Berlin 2017,lwM2sGLJsxo,2017-11-03
3,con|temporary weekends = Famtrips 2.0,eQFzLo2qf40,2017-10-11
4,Meeting Guide Berlin – das Online-Tool für Ver...,VOXBAb7iiGI,2017-06-07
5,"Wenn Sport, dann in Berlin!",3m0oT8sdq5U,2016-12-21
6,Unser Vorsatz für das neue Jahr: Spitzensport ...,LLQ8hnZmiRw,2016-12-21
7,Unser Vorsatz für das neue Jahr: Spitzensport ...,G3pqmV7stqU,2016-12-21
8,MEET+CHANGE,duhJdrAZ1kI,2016-11-22
9,MEET+CHANGE,iXfobx1tdMw,2016-11-11


# **6. CALCULATING OVERALL CHANNEL STATISTICS**

This combines all the 50 Video IDs into a single Comma-Separated string to make a batch call and this minimizes the API resource consumption and the loop iterates through each video to sum the total views, likes and comments.

In [6]:
# Fetching Statistics for All the 50 Videos and Calculating the YouTube Channel Engagement
video_ids_string = ",".join(videos_df["Video ID"])
request = youtube.videos().list(part="statistics", id=video_ids_string)
response = request.execute()
all_views = 0
all_likes = 0
all_comments = 0
for item in response["items"]:
    stats = item["statistics"]
    all_views += int(stats.get("viewCount", 0))
    all_likes += int(stats.get("likeCount", 0))
    all_comments += int(stats.get("commentCount", 0))
# Calculating the Total Channel Engagement Percentage
channel_engagement = ((all_likes + all_comments) / all_views) * 100 if all_views > 0 else 0
print("YOUTUBE CHANNEL STATISTICS")
print("____________________________")
print("The Number of Videos:", len(videos_df))
print("The Number of Views:", all_views)
print("The Number of Likes:", all_likes)
print("The Number of Comments:", all_comments)
print("The Engagement Rate:", round(channel_engagement, 2), "%")

YOUTUBE CHANNEL STATISTICS
____________________________
The Number of Videos: 50
The Number of Views: 278510
The Number of Likes: 594
The Number of Comments: 46
The Engagement Rate: 0.23 %


# **7. DATA AGGREGATION & MONTHLY GROUPING**

Now here we are combining the Detailed Views, the Likes and the Comment Statistics into the Main Video Datasets and then grouping the publication dates by Month and Year, (for example, Sep 2026) to calculate the total monthly views and ensuring that the months stay in the correct chronological order before displaying the final results.

In [7]:
# Merging Video Statistics and Grouping Views by Upload Month
import pandas as pd
from IPython.display import display, HTML
monthly_data = []
for item in response["items"]:
    v_id = item["id"]
    stats = item["statistics"]
    monthly_data.append({
        "Video ID": v_id,
        "Views": int(stats.get("viewCount", 0)),
        "Likes": int(stats.get("likeCount", 0)),
        "Comments": int(stats.get("commentCount", 0))
    })
stats_df = pd.DataFrame(monthly_data)
videos_df = videos_df.drop(columns=["Views", "Likes", "Comments", "Month"], errors="ignore")
videos_df = videos_df.merge(stats_df, on="Video ID")
# Formatting the Dates into Month-Year Format (For Example: Sep 2026)
videos_df["Parsed Date"] = pd.to_datetime(videos_df["Published Date"])
videos_df["Month"] = videos_df["Parsed Date"].dt.strftime("%b %Y")
# We are Grouping the Total Views per Upload Month
monthly_df = videos_df.groupby("Month", as_index=False)["Views"].sum()
monthly_df["SortKey"] = pd.to_datetime(monthly_df["Month"], format="%b %Y")
monthly_df = monthly_df.sort_values("SortKey").drop(columns=["SortKey"])
display(HTML(monthly_df.to_html(index=False)))

Month,Views
Jan 2010,126618
Feb 2010,5532
Jan 2011,5419
Mar 2011,5386
Aug 2011,5602
Sep 2011,739
Dec 2011,18691
Apr 2012,7769
Sep 2012,2016
Apr 2013,49116


# **8. BUILDING INTERACTIVE ANALYTICAL DASHBOARD**

Here we are using a Dropdown Selection Control which allows for dynamic switching between the Whole Channel and for its overall Views or the Specific Selected Individual Videos. We also use the KPI Cards, that is the Key Performance Indicator Cards, and dynamically updates the HTML containers which are used for Views, Likes, Comments, Engagement Rate and the Published Date based on the selected video. Here we use Plotly Bar Chart which displays monthly view trends for the whole channel or single month views for an individual video.

In [8]:
# Creating an Interactive Dashboard Using ipywidgets and Plotly
import ipywidgets as widgets
from IPython.display import display, clear_output, HTML
import plotly.express as px
import plotly.graph_objects as go
import pandas as pd
#Dropdown Sellector for the Whole Channel or for the Specific Video Selection
dropdown = widgets.Dropdown(
    options=["Whole Channel"] + list(videos_df["Video Title"]),
    value="Whole Channel",
    description="Select Video:",
    layout=widgets.Layout(width="600px")
)
dashboard_output = widgets.Output()
bottom_layout_output = widgets.Output()
def update_dashboard(change=None):
    selected = dropdown.value
    #Whole Channel's Aggregated View
    if selected == "Whole Channel":
        views = all_views
        likes = all_likes
        comments = all_comments
        engagement = channel_engagement
        detail_title = "All Tracked Videos"
        detail_date = "Whole Channel"
        chart_data = monthly_df.copy()
        chart_title = "Monthly Views"
        #Individual Selected Video
    else:
        row = videos_df[videos_df["Video Title"] == selected].iloc[0]
        views = int(row["Views"])
        likes = int(row["Likes"])
        comments = int(row["Comments"])
        engagement = ((likes + comments) / views) * 100 if views > 0 else 0
        published = pd.to_datetime(row["Published Date"]).strftime("%d/%m/%Y")
        detail_title = row["Video Title"]
        detail_date = published
        video_month = pd.to_datetime(row["Published Date"]).strftime("%b %Y")
        chart_data = pd.DataFrame({
            "Month": [video_month],
            "Views": [views]
        })
        chart_title = "Monthly Views"
    views_text = f"{views:,}"
    likes_text = f"{likes:,}"
    comments_text = f"{comments:,}"
    engagement_text = f"{engagement:.2f}%"
    #Generate the Key Performance Indicator (KPI) Card Layout
    with dashboard_output:
        clear_output(wait=True)
        top_dashboard = f"""
        <div style="font-family: Arial, sans-serif; background: #ffffff; padding: 20px; border-radius: 12px; border: 1px solid #e2e8f0; width: 900px;">
            <h2 style="margin: 0; color: #172554; font-size: 24px;">visitBerlin YouTube Dashboard</h2>
            <p style="color: #64748b; margin-top: 4px; font-size: 13px;">Explore the performance of visitBerlin's YouTube channel</p>
            <div style="display: flex; gap: 12px; margin-top: 15px;">
                <div style="flex: 1; background: #fff0f3; padding: 14px; border-radius: 8px; border-left: 4px solid #ec4899;">
                    <div style="font-size: 11px; color: #64748b; font-weight: bold;">Views</div>
                    <div style="font-size: 20px; font-weight: bold; color: #be185d; margin-top: 4px;">{views_text}</div>
                </div>
                <div style="flex: 1; background: #fff0f8; padding: 14px; border-radius: 8px; border-left: 4px solid #d946ef;">
                    <div style="font-size: 11px; color: #64748b; font-weight: bold;">Likes</div>
                    <div style="font-size: 20px; font-weight: bold; color: #db2777; margin-top: 4px;">{likes_text}</div>
                </div>
                <div style="flex: 1; background: #fff9e6; padding: 14px; border-radius: 8px; border-left: 4px solid #f59e0b;">
                    <div style="font-size: 11px; color: #64748b; font-weight: bold;">Comments</div>
                    <div style="font-size: 20px; font-weight: bold; color: #d97706; margin-top: 4px;">{comments_text}</div>
                </div>
                <div style="flex: 1; background: #f0f4ff; padding: 14px; border-radius: 8px; border-left: 4px solid #3b82f6;">
                    <div style="font-size: 11px; color: #64748b; font-weight: bold;">Engagement Rate</div>
                    <div style="font-size: 20px; font-weight: bold; color: #2563eb; margin-top: 4px;">{engagement_text}</div>
                </div>
                <div style="flex: 1; background: #eefaff; padding: 14px; border-radius: 8px; border-left: 4px solid #06b6d4;">
                    <div style="font-size: 11px; color: #64748b; font-weight: bold;">Published Date</div>
                    <div style="font-size: 15px; font-weight: bold; color: #0891b2; margin-top: 4px;">{detail_date}</div>
                </div>
            </div>
        </div>
        """
        display(HTML(top_dashboard))
        # Display Interactive Plotly Bar Chart
    fig = px.bar(chart_data, x="Month", y="Views", title=chart_title)
    fig.update_traces(marker_color="#ec4899")
    fig.update_layout(
        height=320,
        width=450,
        plot_bgcolor="white",
        paper_bgcolor="white",
        title_font_size=18,
        title_font_color="#ec4899",
        xaxis_title="Upload Month",
        yaxis_title="",
        margin=dict(l=40, r=20, t=50, b=40)
    )
    fig_widget = go.FigureWidget(fig)
    # Structured HTML Tabular Breakdown
    details_table_html = f"""
    <div style="font-family: Arial, sans-serif; background: #ffffff; padding: 16px; border-radius: 8px; border: 1px solid #dbeafe; width: 400px; height: 320px; box-sizing: border-box;">
        <h3 style="margin-top: 0; color: #2563eb; font-size: 18px; border-bottom: 1px solid #e2e8f0; padding-bottom: 8px;">Video Details</h3>
        <table style="width: 100%; font-size: 13px; border-collapse: collapse; margin-top: 10px;">
            <tr><td style="padding: 8px 0; color: #64748b;">Title</td><td style="padding: 8px 0; font-weight: bold; color: #1e293b; text-align: right;">{detail_title}</td></tr>
            <tr><td style="padding: 8px 0; color: #64748b;">Published Date</td><td style="padding: 8px 0; color: #1e293b; text-align: right;">{detail_date}</td></tr>
            <tr><td style="padding: 8px 0; color: #64748b;">Views</td><td style="padding: 8px 0; color: #1e293b; text-align: right;">{views_text}</td></tr>
            <tr><td style="padding: 8px 0; color: #64748b;">Likes</td><td style="padding: 8px 0; color: #1e293b; text-align: right;">{likes_text}</td></tr>
            <tr><td style="padding: 8px 0; color: #64748b;">Comments</td><td style="padding: 8px 0; color: #1e293b; text-align: right;">{comments_text}</td></tr>
            <tr><td style="padding: 8px 0; color: #64748b;">Engagement Rate</td><td style="padding: 8px 0; color: #1e293b; text-align: right;">{engagement_text}</td></tr>
        </table>
    </div>
    """
    with bottom_layout_output:
        clear_output(wait=True)
        display(widgets.HBox([fig_widget, widgets.HTML(details_table_html)]))
        # Attaching the Event Trigger on Dropdown Selection Change
dropdown.observe(update_dashboard, names="value")
# Display A Box Layout Containing the Controls and Outputs
display(
    widgets.VBox([
        dropdown,
        dashboard_output,
        bottom_layout_output
    ])
)
update_dashboard()

# **9. ENABLING COLAB WIDGET MANAGER**

Since Google Colab restricts custom interactive JavaScript widgets by default, we are permitting the ipywidgets and the plotly figure widget graphics to display and respond interactively within the Web Notebook Interface Environment.

In [ ]:
# Enabling custom third-party JavaScript widgets in Google Colab
from google.colab import output
output.enable_custom_widget_manager()

Support for third party widgets will remain active for the duration of the session. To disable support: